In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path

VCF_PATH   = Path("/home/andrew.dickson/rice_data/sativas413_msu7_final.vcf")
PHENO_PATH = Path("/home/andrew.dickson/rice_data/RiceDiversity_44K_Phenotypes_34traits_PLINK.txt")


def load_vcf_sparse(vcf_path: Path):
    samples, snp_ids = [], []
    rows, cols = [], []
    n_missing = 0
    _gt_map = {"0/0": 0, "1/1": 1, "./.": 0, "0/1": 0.5}

    with open(vcf_path) as fh:
        for line in fh:
            if line.startswith("##"):
                continue
            fields = line.rstrip("\n").split("\t")
            if line.startswith("#CHROM"):
                samples = fields[9:]
                continue
            snp_idx = len(snp_ids)
            snp_ids.append(fields[2])
            gts = fields[9:]
            for samp_idx, gt in enumerate(gts):
                val = _gt_map.get(gt, 0)
                if gt == "./.":
                    n_missing += 1
                if val == 1:
                    rows.append(samp_idx)
                    cols.append(snp_idx)

    X = sp.csr_matrix(
        (np.ones(len(rows), dtype=np.uint8), (rows, cols)),
        shape=(len(samples), len(snp_ids)),
        dtype=np.uint8,
    )
    return X, samples, snp_ids, n_missing


X_vcf, vcf_samples, snp_ids, n_missing = load_vcf_sparse(VCF_PATH)
print(f"Shape (samples × SNPs) : {X_vcf.shape}")
print(f"Stored non-zeros       : {X_vcf.nnz:,}  ({100*X_vcf.nnz/X_vcf.shape[0]/X_vcf.shape[1]:.1f}% density)")
print(f"Missing genotypes (./.): {n_missing:,}")

pheno_raw  = pd.read_csv(PHENO_PATH, sep="\t")
TRAIT_COLS = [c for c in pheno_raw.columns if c not in ("HybID", "NSFTVID")]
print(f"\n{len(TRAIT_COLS)} trait columns.")

EVAL_TRAITS = [
    "Alkali spreading value",
    "Amylose content",
    "Panicle number per plant",
    "Protein content",
    "Seed length",
    "Seed number per panicle",
]

vcf_nsftvid   = [int(s.rsplit("_", 1)[-1]) for s in vcf_samples]
vcf_id_to_idx = {nid: i for i, nid in enumerate(vcf_nsftvid)}

pheno_raw["NSFTVID"] = pheno_raw["NSFTVID"].astype(int)
pheno_matched = pheno_raw[pheno_raw["NSFTVID"].isin(vcf_id_to_idx)].copy().reset_index(drop=True)
print(f"Phenotype rows with VCF match: {len(pheno_matched)} / {len(pheno_raw)}")

vcf_row_order = [vcf_id_to_idx[nid] for nid in pheno_matched["NSFTVID"]]
X = X_vcf[vcf_row_order, :]   # sparse (413, 36900), rows aligned to pheno order
print(f"Aligned X shape: {X.shape}")

Y_raw    = pheno_matched[TRAIT_COLS].values.astype(float)
Y_scaled = np.full_like(Y_raw, np.nan)
for j in range(Y_raw.shape[1]):
    col  = Y_raw[:, j]
    mask = ~np.isnan(col)
    if mask.sum() < 2:
        continue
    mu  = col[mask].mean()
    std = col[mask].std(ddof=0)
    Y_scaled[mask, j] = (col[mask] - mu) / (std if std > 0 else 1.0)

print(f"Y_scaled shape : {Y_scaled.shape}  (samples × traits)")

Shape (samples × SNPs) : (383, 33291)
Stored non-zeros       : 3,215,597  (25.2% density)
Missing genotypes (./.): 378,202

36 trait columns.
Phenotype rows with VCF match: 383 / 413
Aligned X shape: (383, 33291)
Y_scaled shape : (383, 36)  (samples × traits)


In [2]:
from sklearn.decomposition import TruncatedSVD
s = TruncatedSVD(random_state=42, n_components=300)
X_svd = s.fit_transform(X)

In [3]:
import math
from scipy.stats import pearsonr
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.cross_decomposition import PLSRegression


def pearson_correlation_scorer(y_true, y_pred):
    if len(y_true) < 2:
        return 0.0
    r, _ = pearsonr(y_true, y_pred)
    return r if not math.isnan(r) else 0.0


pcc_scorer = make_scorer(pearson_correlation_scorer, greater_is_better=True)
scoring = {
    "MAE": make_scorer(mean_absolute_error, greater_is_better=False),
    "PCC": pcc_scorer,
}

# ── Param grids ───────────────────────────────────────────────────────────────
# TruncatedSVD works natively on sparse matrices; n_components grid mirrors
# the PCA grid used in snp_emb_grid_search.ipynb.

params_rr = {
    "svd__n_components": [50, 100, 200],
    "model__alpha":      np.logspace(-4, 4, 9).tolist(),
}

params_svr = {
    "svd__n_components": [50, 100, 200],
    "model__C":          np.logspace(-2, 3, 6).tolist(),
    "model__gamma":      np.logspace(-4, 1, 6).tolist(),
    "model__kernel":     ["rbf", "poly"],
}

# PLS pipeline: fix SVD at 200 components (covers ~95% variance),
# then grid-search over PLS n_components exactly as in the embedding notebook.
params_pls = {
    "svd__n_components":   [200],
    "model__n_components": list(range(2, 21)),
}

# ── Pipelines ─────────────────────────────────────────────────────────────────

n_splits    = 5
cv_strategy = KFold(n_splits=n_splits, shuffle=True, random_state=42)

pipe_rr  = Pipeline([("svd", TruncatedSVD(random_state=42)), ("model", Ridge())])
pipe_svr = Pipeline([("svd", TruncatedSVD(random_state=42)), ("model", SVR())])
pipe_pls = Pipeline([("svd", TruncatedSVD(random_state=42)), ("model", PLSRegression())])

sklearn_pipelines = [
    ("RR-BLUP/Ridge", pipe_rr,  params_rr),
    ("SVR",           pipe_svr, params_svr),
    # ("Random Forest",     pipe_rf,  params_rf),
    # ("Gradient Boosting", pipe_gbr, params_gbr),
]

print("Imports and config ready.")
print(f"  {len(EVAL_TRAITS)} traits, {n_splits}-fold CV, {len(sklearn_pipelines)} sklearn models + PLS")

Imports and config ready.
  6 traits, 5-fold CV, 2 sklearn models + PLS


In [4]:
results = []

for j, trait in enumerate(EVAL_TRAITS):
    t_idx = TRAIT_COLS.index(trait)
    y_col = Y_scaled[:, t_idx]
    mask  = ~np.isnan(y_col)
    if mask.sum() < n_splits * 2:
        print(f"[{j+1}/{len(EVAL_TRAITS)}] {trait}: skipped (only {mask.sum()} non-NaN samples)")
        continue

    X_t = X_svd[mask]   # sparse slice
    y_t = y_col[mask]

    print(f"\n{'='*60}")
    print(f"[{j+1}/{len(EVAL_TRAITS)}] Trait: {trait}  (n={mask.sum()})")

    best_pcc    = -np.inf
    best_mae    = np.inf
    best_params = None
    best_model  = None

    for name, pipeline, param_grid in sklearn_pipelines:
        gs = GridSearchCV(
            estimator  = pipeline,
            param_grid = param_grid,
            scoring    = scoring,
            refit      = "PCC",
            cv         = cv_strategy,
            n_jobs     = -1,
            verbose    = 0,
        )
        gs.fit(X_t, y_t)

        idx     = gs.best_index_
        pcc_    = gs.cv_results_["mean_test_PCC"][idx]
        mae_    = -gs.cv_results_["mean_test_MAE"][idx]
        params_ = gs.best_params_

        print(f"  {name:<22}  PCC={pcc_:.4f}  MAE={mae_:.4f}  {params_}")

        if pcc_ > best_pcc:
            best_pcc, best_mae, best_params, best_model = pcc_, mae_, params_, name

    # PLS with SVD prefix (PLSRegression cannot ingest sparse directly)
    gs_pls = GridSearchCV(
        estimator  = pipe_pls,
        param_grid = params_pls,
        scoring    = scoring,
        refit      = "PCC",
        cv         = cv_strategy,
        n_jobs     = -1,
        verbose    = 0,
    )
    gs_pls.fit(X_t, y_t)

    idx_pls = gs_pls.best_index_
    pcc_pls = gs_pls.cv_results_["mean_test_PCC"][idx_pls]
    mae_pls = -gs_pls.cv_results_["mean_test_MAE"][idx_pls]

    print(f"  {'PLS':<22}  PCC={pcc_pls:.4f}  MAE={mae_pls:.4f}  {gs_pls.best_params_}")

    if pcc_pls > best_pcc:
        best_pcc, best_mae, best_params, best_model = pcc_pls, mae_pls, gs_pls.best_params_, "PLS"

    print(f"  >> Best: {best_model}  PCC={best_pcc:.4f}  MAE={best_mae:.4f}")

    results.append({
        "Trait":      trait,
        "Model":      best_model,
        "PCC":        round(best_pcc,  4),
        "MAE":        round(best_mae,  4),
        "BestParams": best_params,
        "n_samples":  int(mask.sum()),
    })

results_df = pd.DataFrame(results).sort_values("PCC", ascending=False).reset_index(drop=True)
print("\n\n" + "="*60)
print("FINAL RESULTS (sorted by PCC)")
print(results_df[["Trait", "Model", "PCC", "MAE", "n_samples"]].to_string(index=False))


[1/6] Trait: Alkali spreading value  (n=374)
  RR-BLUP/Ridge           PCC=0.5210  MAE=0.5994  {'model__alpha': 1000.0, 'svd__n_components': 200}
  SVR                     PCC=0.5876  MAE=0.5653  {'model__C': 1.0, 'model__gamma': 0.001, 'model__kernel': 'rbf', 'svd__n_components': 100}
  PLS                     PCC=0.4909  MAE=0.6336  {'model__n_components': 4, 'svd__n_components': 200}
  >> Best: SVR  PCC=0.5876  MAE=0.5653

[2/6] Trait: Amylose content  (n=372)
  RR-BLUP/Ridge           PCC=0.8017  MAE=0.4162  {'model__alpha': 1000.0, 'svd__n_components': 200}
  SVR                     PCC=0.8072  MAE=0.4093  {'model__C': 10.0, 'model__gamma': 9.999999999999999e-05, 'model__kernel': 'rbf', 'svd__n_components': 200}
  PLS                     PCC=0.7980  MAE=0.4113  {'model__n_components': 2, 'svd__n_components': 200}
  >> Best: SVR  PCC=0.8072  MAE=0.4093

[3/6] Trait: Panicle number per plant  (n=345)
  RR-BLUP/Ridge           PCC=0.8247  MAE=0.4426  {'model__alpha': 1000.0, 'svd__n

/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation c

  SVR                     PCC=0.4790  MAE=0.6856  {'model__C': 0.09999999999999999, 'model__gamma': 0.001, 'model__kernel': 'rbf', 'svd__n_components': 100}
  PLS                     PCC=0.3941  MAE=0.7246  {'model__n_components': 2, 'svd__n_components': 200}
  >> Best: SVR  PCC=0.4790  MAE=0.6856

[5/6] Trait: Seed length  (n=350)
  RR-BLUP/Ridge           PCC=0.7464  MAE=0.4835  {'model__alpha': 1000.0, 'svd__n_components': 200}


/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y

  SVR                     PCC=0.7339  MAE=0.4875  {'model__C': 100.0, 'model__gamma': 9.999999999999999e-05, 'model__kernel': 'rbf', 'svd__n_components': 200}
  PLS                     PCC=0.7452  MAE=0.4809  {'model__n_components': 2, 'svd__n_components': 200}
  >> Best: RR-BLUP/Ridge  PCC=0.7464  MAE=0.4835

[6/6] Trait: Seed number per panicle  (n=349)
  RR-BLUP/Ridge           PCC=0.5734  MAE=0.6311  {'model__alpha': 1000.0, 'svd__n_components': 200}


/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2486610/1125763216.py:14: NearConstantInputWarning: An input array is nearly constant; the co

  SVR                     PCC=0.5875  MAE=0.6431  {'model__C': 10.0, 'model__gamma': 9.999999999999999e-05, 'model__kernel': 'rbf', 'svd__n_components': 100}
  PLS                     PCC=0.5485  MAE=0.6578  {'model__n_components': 20, 'svd__n_components': 200}
  >> Best: SVR  PCC=0.5875  MAE=0.6431


FINAL RESULTS (sorted by PCC)
                   Trait         Model    PCC    MAE  n_samples
Panicle number per plant RR-BLUP/Ridge 0.8247 0.4426        345
         Amylose content           SVR 0.8072 0.4093        372
             Seed length RR-BLUP/Ridge 0.7464 0.4835        350
  Alkali spreading value           SVR 0.5876 0.5653        374
 Seed number per panicle           SVR 0.5875 0.6431        349
         Protein content           SVR 0.4790 0.6856        364


In [8]:
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
results_df[["Trait", "Model", "PCC", "MAE", "n_samples"]].to_csv(
    RESULTS_DIR / "svd_grid_search_results.csv", index=False
)
print("Saved svd_grid_search_results.csv")

Saved svd_grid_search_results.csv
